In [1]:
import os, random, time, itertools, json, warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib
matplotlib.use('Agg')          
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats        
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import datasets, transforms, models
from sklearn.cluster import MiniBatchKMeans, KMeans
from sklearn.neighbors import NearestNeighbors
from sklearn.manifold import TSNE

print('Imports OK')
print('torch', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

Imports OK
torch 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


In [2]:
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

DEVICE   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SAVE_DIR = '/kaggle/working/typiclust_ckpts'
PLOT_DIR = '/kaggle/working/plots'
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(PLOT_DIR, exist_ok=True)

CFG = {
    # SimCLR self-supervised pre-training
    'simclr_epochs'      : 100, 
    'simclr_lr'          : 0.4,
    'simclr_batch'       : 512,
    'simclr_temperature' : 0.5,
    'simclr_proj_dim'    : 128,
    'simclr_momentum'    : 0.9,
    'simclr_wd'          : 1e-4,

    # Framework A – Fully Supervised
    'sup_epochs'         : 70,
    'sup_lr'             : 0.025,
    'sup_batch'          : 128,
    'sup_momentum'       : 0.9,

    # Framework B – Linear Probe on SimCLR embeddings
    'lp_epochs'          : 50,    # paper: 100  [↓2×]
    'lp_lr'              : 2.5,
    'lp_batch'           : 256,

    # Framework C – Semi-Supervised (FixMatch-lite) 
    'ssl_iters'          : 30_000, 
    'ssl_lr'             : 0.03,
    'ssl_batch_l'        : 64,
    'ssl_batch_u'        : 64,
    'ssl_threshold'      : 0.95,
    'ssl_wd'             : 5e-4,

    # Active-Learning settings
    'al_budget'          : 10, 
    'al_iterations'      : 5,  
    'typi_k'             : 20,   
    'max_clusters'       : 500,  
}

print('Device:', DEVICE)
print('Save dir:', SAVE_DIR)
print('Config loaded. Total label budget:', CFG['al_budget'] * CFG['al_iterations'])

Device: cuda
Save dir: /kaggle/working/typiclust_ckpts
Config loaded. Total label budget: 50


In [3]:
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2470, 0.2435, 0.2616)

tf_plain = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])
tf_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])
tf_weak = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])
tf_strong = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.4, 0.4, 0.4, 0.1),
    transforms.RandomGrayscale(p=0.2),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])
# SimCLR augmentation: random resized crop + colour jitter + grayscale
tf_simclr = transforms.Compose([
    transforms.RandomResizedCrop(32, scale=(0.2, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.4, 0.4, 0.4, 0.1),
    transforms.RandomGrayscale(p=0.2),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])

DATA_ROOT = '/kaggle/working/data'
os.makedirs(DATA_ROOT, exist_ok=True)

raw_train = datasets.CIFAR10(DATA_ROOT, train=True,  download=True)
raw_test  = datasets.CIFAR10(DATA_ROOT, train=False, download=True)

N_TRAIN       = len(raw_train)   # 50 000
N_TEST        = len(raw_test)    # 10 000
N_CLS         = 10
ALL_TRAIN_IDX = list(range(N_TRAIN))
CLASSES       = raw_train.classes 

class PlainDataset(Dataset):
    def __init__(self, raw, transform):
        self.raw = raw
        self.tf  = transform
    def __len__(self):
        return len(self.raw)
    def __getitem__(self, i):
        img, label = self.raw[i]
        return self.tf(img), label

test_loader = DataLoader(
    PlainDataset(raw_test, tf_plain),
    batch_size=512, shuffle=False, num_workers=2, pin_memory=True,
)
print(f'Train: {N_TRAIN}  Test: {N_TEST}  Classes: {CLASSES}')

100%|██████████| 170M/170M [00:06<00:00, 27.0MB/s]


Train: 50000  Test: 10000  Classes: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


In [4]:
class SimCLRModel(nn.Module):
    """ResNet-18 backbone + two-layer MLP projector (Chen et al. 2020).
    For 32×32 CIFAR: replace 7×7 conv with 3×3 and remove maxpool.
    """
    def __init__(self, proj_dim: int = 128):
        super().__init__()
        self.backbone = models.resnet18(weights=None)
        # CIFAR adaptation (paper Appendix F.1)
        self.backbone.conv1   = nn.Conv2d(3, 64, 3, 1, 1, bias=False)
        self.backbone.maxpool = nn.Identity()
        feat_dim = self.backbone.fc.in_features  # 512
        self.backbone.fc = nn.Identity()
        # Two-layer MLP projector
        self.projector = nn.Sequential(
            nn.Linear(feat_dim, feat_dim),
            nn.ReLU(inplace=True),
            nn.Linear(feat_dim, proj_dim),
        )

    def forward(self, x, return_feat: bool = False):
        feat = self.backbone(x)          # (N, 512)
        if return_feat:
            return F.normalize(feat, dim=1)   # L2-normalised 512-d embedding
        return F.normalize(self.projector(feat), dim=1)  # 128-d projection


class NTXentLoss(nn.Module):
    """Normalised temperature-scaled cross-entropy (NT-Xent, Chen et al. 2020)."""
    def __init__(self, temperature: float = 0.5):
        super().__init__()
        self.T = temperature

    def forward(self, z1, z2):
        N = z1.size(0)
        z   = torch.cat([z1, z2], dim=0)                    # (2N, D)
        sim = torch.mm(z, z.t()) / self.T                   # (2N, 2N)
        sim.masked_fill_(
            torch.eye(2 * N, dtype=torch.bool, device=z.device), float('-inf'))
        pos = torch.cat([torch.arange(N, 2*N),
                          torch.arange(N)]).to(z.device)
        return F.cross_entropy(sim, pos)


class SimCLRDataset(Dataset):
    """Returns two augmented views of the same image."""
    def __init__(self, raw):
        self.raw = raw
    def __len__(self):
        return len(self.raw)
    def __getitem__(self, i):
        img, _ = self.raw[i]
        return tf_simclr(img), tf_simclr(img)

print('SimCLR model definition ready.')

SimCLR model definition ready.


In [5]:
SIMCLR_CKPT  = os.path.join(SAVE_DIR, 'simclr_model.pt')
EMBED_PATH   = os.path.join(SAVE_DIR, 'embeddings.npy')
SIMCLR_LOG   = os.path.join(SAVE_DIR, 'simclr_loss_log.json')

def train_simclr():
    if os.path.exists(SIMCLR_CKPT) and os.path.exists(EMBED_PATH):
        print('SimCLR checkpoint found – skipping training.')
        return

    epochs = CFG['simclr_epochs']
    print(f'Training SimCLR for {epochs} epochs on {DEVICE} ...')
    t0 = time.time()

    loader = DataLoader(
        SimCLRDataset(raw_train),
        batch_size=CFG['simclr_batch'], shuffle=True,
        num_workers=2, pin_memory=True, drop_last=True,
    )
    model     = SimCLRModel(CFG['simclr_proj_dim']).to(DEVICE)
    criterion = NTXentLoss(CFG['simclr_temperature'])
    optimizer = optim.SGD(
        model.parameters(),
        lr=CFG['simclr_lr'],
        momentum=CFG['simclr_momentum'],
        weight_decay=CFG['simclr_wd'],
    )
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    loss_log = []
    for ep in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        for v1, v2 in loader:
            v1, v2 = v1.to(DEVICE), v2.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(v1), model(v2))
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        scheduler.step()
        avg_loss = total_loss / len(loader)
        loss_log.append(avg_loss)
        if ep % 10 == 0 or ep == 1:
            print(f'  Ep {ep:3d}/{epochs}  loss={avg_loss:.4f}  '
                  f'lr={scheduler.get_last_lr()[0]:.4f}  '
                  f't={time.time()-t0:.0f}s')

    # Save model checkpoint
    torch.save({
        'model_state': model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'epochs_trained': epochs,
        'cfg': {k: v for k, v in CFG.items() if 'simclr' in k},
    }, SIMCLR_CKPT)
    json.dump(loss_log, open(SIMCLR_LOG, 'w'))
    print(f'Checkpoint saved -> {SIMCLR_CKPT}')

    # ── Extract and save 512-d L2-normalised embeddings ──────────────────────
    print('Extracting embeddings for all 50 000 training images ...')
    model.eval()
    embed_loader = DataLoader(
        PlainDataset(raw_train, tf_plain),
        batch_size=512, shuffle=False, num_workers=2, pin_memory=True,
    )
    feats = []
    with torch.no_grad():
        for x, _ in embed_loader:
            feats.append(model(x.to(DEVICE), return_feat=True).cpu().numpy())
    embs = np.concatenate(feats, axis=0)   # (50000, 512)
    np.save(EMBED_PATH, embs)
    print(f'Embeddings saved -> {EMBED_PATH}  shape={embs.shape}  '
          f'total_time={time.time()-t0:.0f}s')


train_simclr()

Training SimCLR for 100 epochs on cuda ...
  Ep   1/100  loss=6.3137  lr=0.3999  t=80s
  Ep  10/100  loss=5.4906  lr=0.3902  t=867s
  Ep  20/100  loss=5.3967  lr=0.3618  t=1741s
  Ep  30/100  loss=5.3493  lr=0.3176  t=2614s
  Ep  40/100  loss=5.3231  lr=0.2618  t=3486s
  Ep  50/100  loss=5.2941  lr=0.2000  t=4356s
  Ep  60/100  loss=5.2731  lr=0.1382  t=5227s
  Ep  70/100  loss=5.2520  lr=0.0824  t=6099s
  Ep  80/100  loss=5.2316  lr=0.0382  t=6971s
  Ep  90/100  loss=5.2145  lr=0.0098  t=7843s
  Ep 100/100  loss=5.2074  lr=0.0000  t=8713s
Checkpoint saved -> /kaggle/working/typiclust_ckpts/simclr_model.pt
Extracting embeddings for all 50 000 training images ...
Embeddings saved -> /kaggle/working/typiclust_ckpts/embeddings.npy  shape=(50000, 512)  total_time=8725s
